# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [ ]:
# ===== Setup Colab tu GitHub =====
import os
import sys
import subprocess

from google.colab import drive

drive.mount("/content/drive")

MSSV = "2A202602364"
BRANCH = "submission-2A202602364"
REPO_URL = "https://github.com/n4hhh/K4-Track4-Day1-Neuralnetwork.git"
REPO_ROOT = "/content/K4-Track4-Day1-Neuralnetwork"

if not os.path.isdir(f"{REPO_ROOT}/.git"):
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, REPO_ROOT],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", REPO_ROOT, "pull", "--ff-only"],
        check=True,
    )

CODE_DIR = f"{REPO_ROOT}/submission_{MSSV}/code"
OUT_DIR = (
    f"/content/drive/MyDrive/AICB2026/"
    f"K4-Track4-Day1-Neuralnetwork/submission_{MSSV}"
)

os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Code directory:", CODE_DIR)
print("PyTorch:", torch.__version__)
print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

assert device == "cuda", "Hay chon Runtime > Change runtime type > T4 GPU"

## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
from data import prepare_data

subprocess.run(
    [sys.executable, "scripts/split_data.py"],
    cwd=REPO_ROOT,
    check=True,
)

data = prepare_data(
    device=device,
    val_fraction=0.2,
    seed=42,
    processed_dir=f"{REPO_ROOT}/data/processed",
)

assert tuple(data["X_tr"].shape) == (371_847, 54)
assert tuple(data["X_val"].shape) == (92_962, 54)
assert tuple(data["X_eval"].shape) == (116_203, 54)

assert data["X_tr"].dtype == torch.float32
assert data["X_val"].dtype == torch.float32
assert data["X_eval"].dtype == torch.float32
assert data["y_tr"].dtype == torch.int64

for name in ("y_tr", "y_val", "y_eval"):
    counts = torch.bincount(data[name], minlength=7).double()
    proportions = counts / counts.sum()
    print(
        f"{name} class proportions:",
        proportions.cpu().numpy().round(5),
    )

numeric_train = data["X_tr"][:, :10].double()
numeric_mean = numeric_train.mean(dim=0).cpu()
numeric_std = numeric_train.std(dim=0, correction=0).cpu()

binary_values = torch.unique(data["X_tr"][:, 10:]).cpu().tolist()

print("Mean 10 numeric columns:", numeric_mean.numpy().round(6))
print("Std 10 numeric columns:", numeric_std.numpy().round(6))
print("One-hot values:", binary_values)

assert torch.allclose(
    numeric_mean,
    torch.zeros_like(numeric_mean),
    atol=1e-5,
)
assert torch.allclose(
    numeric_std,
    torch.ones_like(numeric_std),
    atol=1e-5,
)
assert binary_values == [0.0, 1.0]

print("Part 0: PASS")

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import torch.nn.functional as F

from model import (
    MLP,
    EXPECTED_PARAMS,
    activation_stats,
    count_params,
)

# Dat seed de ket qua co the lap lai
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)


# 1. Kiem tra kien truc va so tham so
model = MLP(
    hidden=(256, 128),
    dropout=0.0,
    init="he",
).to(device)

n_params = count_params(model)

assert n_params == EXPECTED_PARAMS[(256, 128)]
assert n_params == 47_879

print("M-base parameters:", n_params)


# 2. Kiem tra shape logits
model.eval()

with torch.no_grad():
    probe_logits = model(data["X_val"][:8])

assert tuple(probe_logits.shape) == (8, 7)

print("Input shape:", tuple(data["X_val"][:8].shape))
print("Logits shape:", tuple(probe_logits.shape))


# 3. Thong ke kich hoat
stats = activation_stats(
    model,
    data["X_val"][:512],
)

print(
    "Activation std after Linear layers:",
    [round(value, 4) for value in stats],
)


# 4. Loss buoc 0 tren validation
with torch.no_grad():
    step0_logits = model(data["X_val"])
    step0_loss = F.cross_entropy(
        step0_logits,
        data["y_val"],
    ).item()

expected_random_loss = math.log(7)

print(f"Step-0 CE loss: {step0_loss:.4f}")
print(f"ln(7): {expected_random_loss:.4f}")
print(
    "Absolute difference:",
    round(abs(step0_loss - expected_random_loss), 4),
)

assert math.isfinite(step0_loss)


# 5. Kiem tra gradient chay toi moi tham so
model.zero_grad(set_to_none=True)

gradient_logits = model(data["X_tr"][:512])
gradient_loss = F.cross_entropy(
    gradient_logits,
    data["y_tr"][:512],
)

gradient_loss.backward()

for name, parameter in model.named_parameters():
    grad_norm = None if parameter.grad is None else parameter.grad.norm().item()

    print(f"Gradient {name}: {grad_norm:.6f}")

    assert grad_norm is not None
    assert grad_norm > 0.0


# 6. Thu qua khop 20 mau
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

tiny_model = MLP(
    hidden=(256, 128),
    dropout=0.0,
    init="he",
).to(device)

tiny_optimizer = torch.optim.Adam(
    tiny_model.parameters(),
    lr=1e-2,
)

tiny_X = data["X_tr"][:20]
tiny_y = data["y_tr"][:20]
tiny_losses = []

tiny_model.train()

for step in range(501):
    tiny_optimizer.zero_grad(set_to_none=True)

    tiny_logits = tiny_model(tiny_X)
    tiny_loss = F.cross_entropy(
        tiny_logits,
        tiny_y,
    )

    tiny_losses.append(tiny_loss.item())

    if step == 500:
        break

    tiny_loss.backward()
    tiny_optimizer.step()


tiny_model.eval()

with torch.no_grad():
    tiny_predictions = tiny_model(tiny_X).argmax(dim=1)

    tiny_accuracy = (tiny_predictions == tiny_y).float().mean().item()

print(
    f"Overfit 20 samples: "
    f"loss={tiny_losses[-1]:.6f}, "
    f"accuracy={tiny_accuracy:.1%}"
)

assert tiny_losses[-1] < 0.02
assert tiny_accuracy == 1.0


# 7. Luu duong cong overfit
figure_path = f"{OUT_DIR}/figures/part1_overfit20.png"

plt.figure(figsize=(7, 4))
plt.plot(tiny_losses)
plt.yscale("log")
plt.xlabel("Training step")
plt.ylabel("Cross-entropy loss")
plt.title("Part 1 health check: overfit 20 samples")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(
    figure_path,
    dpi=160,
    bbox_inches="tight",
)
plt.show()

print("Saved figure:", figure_path)
print("Part 1: PASS")

**Nhận xét Part 1 (tự viết):** loss bước 0 đo được = ___, so với ln 7 = 1.946 vì sao ...; quá khớp 20 mẫu: ...

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [ ]:
# TODO: chọn lr cho baseline bằng VAL (chạy vài giá trị, ví dụ quanh 0.01 -> 0.1); ghi bảng lr -> val_macro_f1
# TODO: cfg = {**DEFAULT_CFG, "lr": <lr đã chọn>}
# TODO: chạy baseline với >= 2 seed (khuyến khích 3): result = run_experiment({**cfg, "seed": s, "exp_id": f"base-s{s}"}, data)
# TODO: với mỗi result: save_result(result, f"{OUT_DIR}/results"); plot_run(result, f"{OUT_DIR}/figures/{exp_id}.png")
# TODO: tính trung bình ± độ lệch chuẩn (val_macro_f1, val_acc) qua các seed -> ngưỡng nhiễu 2σ

**Nhận xét baseline (tự viết):** hình dạng đường cong (còn giảm? bắt đầu quá khớp?), val_acc so với mốc "đoán đa số", độ nhiễu giữa các seed ...

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Thí nghiệm `<exp_id>` — chủ đề `<nhóm>`
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
# Mẫu một thí nghiệm. Lặp lại / bọc trong vòng for qua danh sách cfg bạn tự thiết kế.
# TODO: exp_cfg = {**cfg, "exp_id": "...", "group": "...", "description": "...", <chỉ đổi MỘT yếu tố>}
# TODO: result = run_experiment(exp_cfg, data)
# TODO: save_result(...); plot_run(...)   # mỗi thí nghiệm một ảnh figures/<exp_id>.png
# TODO: in summary (best_val_loss, val_acc, val_macro_f1, ...) và so với baseline + ngưỡng nhiễu 2σ

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

In [ ]:
# TODO: ảnh chồng theo nhóm: plot_compare(list_of_results_in_group, "val_loss", f"{OUT_DIR}/figures/compare_<nhóm>.png")

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [ ]:
# TODO: chọn cấu hình cuối cùng theo VAL (có thể là baseline nếu bạn không kết hợp thêm gì)
# TODO: final_eval(cfg_final, result_final, data, f"{OUT_DIR}/predictions_eval.csv")
# TODO: chạy: python scripts/evaluate.py --pred <OUT_DIR>/predictions_eval.csv --out <OUT_DIR>/eval_result.json (từ REPO_ROOT)
# TODO: đọc eval_result.json: ghi accuracy, macro_f1 vào bảng và báo cáo; làm tương tự cho baseline (predictions riêng, không nộp nếu không cần)

In [ ]:
# TODO: phân tích lỗi trên eval: bảng precision/recall/F1 theo lớp (từ eval_result.json) và ma trận nhầm lẫn;
#       lớp nào khó nhất? vì sao (số mẫu ít? nhầm với lớp nào?)  -> viết nhận xét ở ô markdown bên dưới

**Phân tích lỗi (tự viết):** ...

In [ ]:
# TODO: results = load_results(f"{OUT_DIR}/results"); rows = [to_row(r, ...) for r in results]
# TODO: write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
# TODO: bổ sung sheet Seeds (exp_id các lần chạy baseline) và nhận xét ở sheet Summary